## This notebook serves as a demonstration of the LASP SDTP application

#### Some commands to make things easier to test within this notebook

In [1]:
import json
from lasp_sdtp.database.database_controller import db
from tests import conftest

# Clear out contents of database
db.session.query(db.FileQueue).delete()
db.session.query(db.Transactions).delete()
db.session.query(db.Accounts).delete()
db.session.query(db.Metadata).delete()
db.session.query(db.AvailableFiles).delete()
db.session.commit()

# Insert an account to test with
db.insert_data('accounts', [{
    'username': 'ges_disc',
    'role': 'subscriber',
    'allowed_data_products': 'tsis2',
    'registration_open': True}])

# Add files to database
conftest._add_available_files_entries()

## Configuration

#### Subscribers must configure several parameters prior to using the `lasp_sdtp` application.  To configure a subscriber, instantiate the SubsciberConfig class, set the necessary parameters, and save.

In [2]:
from lasp_sdtp.config import SubscriberConfig

In [3]:
config = SubscriberConfig()

config.account_expiration_period = 1800  # The number of days the subscriber account will be valid
config.checksum_type = 'sha256'  # The type of checksum that the subscriber will use to validate responses
config.distinguished_name = ''  # The distingushed name to use in authentication
config.expiration_period = 180  # The number of days files will remain in the file queue
config.max_num_files = 10000  # The maximum number of files to return in file list requests
config.num_download_threads = 5  # The maximum number of simultaneous downloads
config.username = 'ges_disc'  # The subscriber username (used in authentication)

# Define the file stream(s) used for file transfer, along with any unique extras and tags
config.streams = {
    "prod": {
        "extras": {"type": {"type": "str", "default": ""}},
        "tags": {"date": {"type": "str", "default": ""}}
    },
    "dev": {
        "extras": {"was_extrapolated": {"type": "bool", "default": ""}},
        "tags": {"irradiance": {"type": "float", "default": ""}}
    }
}

In [4]:
config.__dict__

{'account_expiration_period': 1800,
 'checksum_type': 'sha256',
 'distinguished_name': '',
 'expiration_period': 180,
 'max_num_files': 10000,
 'num_download_threads': 5,
 'username': 'ges_disc',
 'streams': {'prod': {'extras': {'type': {'type': 'str', 'default': ''}},
   'tags': {'date': {'type': 'str', 'default': ''}}},
  'dev': {'extras': {'was_extrapolated': {'type': 'bool', 'default': ''}},
   'tags': {'irradiance': {'type': 'float', 'default': ''}}}}}

In [5]:
config.save()

Checking to see if config is valid
Configuration is valid.
data/subscriber_config.json already exists. Are you sure you want to overwrite? (y/n)
y

Configuration file saved to data/subscriber_config.json


## Register Account

Register a new account.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned.
    
Command: `curl PUT "http://127.0.0.1:8000/register -H "Cert-UID: <certificate>"`

In [6]:
! curl -i -X PUT "http://127.0.0.1:8000/register" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## List Available Files

Return a list of available files given user-supplied parameters.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `200` response is returned with a list of the available files and their metadata.  If no parameters are supplied, then all available files are returned.

Command: `curl GET "http://127.0.0.1:8000/files?param1=value1&param2=value2... -H "Cert-UID: <certificate>"`

#### Supported parameters:
- `stream` (e.g. `prod`)
- `ShortName` (e.g. `TSIS_SC_L2`)
- `version` (e.g. `v01`)
- `date` (e.g.`2022-01-01`)
- `start_date` (e.g. `2022-01-01`)
- `end_date` (e.g. `2022-01-02`)


Note that `date` can only be supplied if `start_date` and/or `end_date` are *not* supplied.  Also, if `start_date` is supplied, then `end_date` *must* be supplied, or vice-versa 

#### Examples

In [7]:
# List all available files
! curl -i -X GET "http://127.0.0.1:8000/files" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "checksum": "sha256:7qe3m0swbuom2tvc3tzethdyj086nvtale55rjao2tc6foq6wtcc7gqr9qpw9q4j", 
      "data_product_id": "tsis2", 
      "date": "2021-12-31", 
      "expires": "2023-03-07", 
      "fileid": 1768, 
      "name": "tsis2_sc_L2_v01_20220102_20220103.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:0azfeqqyp71aydrpia34k0cpxo2qffh1e2ku9e5j0l1c2dd2uzuip84nrww99rx5", 
      "data_product_id": "tsis2", 
      "date": "2022-01-01", 
      "expires": "2023-03-07", 
      "fileid": 1769, 
      "name": "tsis2_sc_L2_v01_20220103_20220104.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 

    }, 
    {
      "checksum": "sha256:rpn6hftc9opfcy35ohp9igxxyaolxh26buh24ruo6gd0u1pnq9229zgiemajxojl", 
      "data_product_id": "tsis2", 
      "date": "2022-01-02", 
      "expires": "2023-03-07", 
      "fileid": 1770, 
      "na

In [8]:
# List all available files with ShortName of 'TSIS_SC_L2'
! curl -i -X GET "http://127.0.0.1:8000/files?ShortName=TSIS_SC_L2" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "checksum": "sha256:7qe3m0swbuom2tvc3tzethdyj086nvtale55rjao2tc6foq6wtcc7gqr9qpw9q4j", 
      "data_product_id": "tsis2", 
      "date": "2021-12-31", 
      "expires": "2023-03-07", 
      "fileid": 1768, 
      "name": "tsis2_sc_L2_v01_20220102_20220103.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:0azfeqqyp71aydrpia34k0cpxo2qffh1e2ku9e5j0l1c2dd2uzuip84nrww99rx5", 
      "data_product_id": "tsis2", 
      "date": "2022-01-01", 
      "expires": "2023-03-07", 
      "fileid": 1769, 
      "name": "tsis2_sc_L2_v01_20220103_20220104.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:rpn6hftc9opfcy35ohp9igxxyaolxh26buh24ruo6gd0u1pnq9229zgiemajxojl", 
      "data_product_id": "tsis2", 
      "date": "2022-01-02", 
      "expires": "2023-03-07", 
      "fil

In [9]:
# List all available files with specific ShortName and for specific date
! curl -i -X GET "http://127.0.0.1:8000/files?ShortName=TSIS_SC_L2&date=2022-01-02" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "checksum": "sha256:rpn6hftc9opfcy35ohp9igxxyaolxh26buh24ruo6gd0u1pnq9229zgiemajxojl", 
      "data_product_id": "tsis2", 
      "date": "2022-01-02", 
      "expires": "2023-03-07", 
      "fileid": 1770, 
      "name": "tsis2_sc_L2_v01_20220104_20220105.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }
  ]
}


In [10]:
# List all available files between range between dates
! curl -i -X GET "http://127.0.0.1:8000/files?start_date=2021-12-31&end_date=2022-01-05" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": [
    {
      "checksum": "sha256:7qe3m0swbuom2tvc3tzethdyj086nvtale55rjao2tc6foq6wtcc7gqr9qpw9q4j", 
      "data_product_id": "tsis2", 
      "date": "2021-12-31", 
      "expires": "2023-03-07", 
      "fileid": 1768, 
      "name": "tsis2_sc_L2_v01_20220102_20220103.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:0azfeqqyp71aydrpia34k0cpxo2qffh1e2ku9e5j0l1c2dd2uzuip84nrww99rx5", 
      "data_product_id": "tsis2", 
      "date": "2022-01-01", 
      "expires": "2023-03-07", 
      "fileid": 1769, 
      "name": "tsis2_sc_L2_v01_20220103_20220104.zip", 
      "shortname": "TSIS_SC_L2", 
      "size": 55.0, 
      "stream": "prod", 
      "version": "v01"
    }, 
    {
      "checksum": "sha256:rpn6hftc9opfcy35ohp9igxxyaolxh26buh24ruo6gd0u1pnq9229zgiemajxojl", 
      "data_product_id": "tsis2", 
      "date": "2022-01-02", 
      "expires": "2023-03-07", 
      "fil

## Get Specific File

Return the contents of a file for a given `fileid`.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `200` response is returned with the content of the file.  

Command: `curl GET "http://127.0.0.1:8000/files/<fileid> -H "Cert-UID: <certificate>"`

In [11]:
! curl -i -X GET "http://127.0.0.1:8000/files/1810" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "contents": [
    "File contents for tsis2_sc_L2_v01_20220101_20220102.zip"
  ], 
  "filename": "tsis2_sc_L2_v01_20220101_20220102.zip"
}


## Delete Specific File

Remove a specific file from the queue.  The user must supply the `Cert-UID` in the header of the request.  If successful, a `204` response is returned with the content of the file.  

Command: `curl DELETE "http://127.0.0.1:8000/files/<fileid> -H "Cert-UID: <certificate>"`

In [12]:
! curl -i -X DELETE "http://127.0.0.1:8000/files/1810" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## Delete Range of Files

Remove a range of files from the queue.  The user must supply the `Cert-UID` in the header of the request.  The range of files should be positive integers that increase in value.  If successful, a `204` response is returned.

Command: `curl DELETE "http://127.0.0.1:8000/files/<fileid_start>-<fileid_end> -H "Cert-UID: <certificate>"`

In [13]:
# First get a few files so that they are in the queue
! curl -i -X GET "http://127.0.0.1:8000/files/1810" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"
! curl -i -X GET "http://127.0.0.1:8000/files/1811" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"
! curl -i -X GET "http://127.0.0.1:8000/files/1812" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

HTTP/1.0 200 OK
Content-Type: application/json
Content-Length: 142
SDTP-TransactionID: 720
Server: Werkzeug/2.0.3 Python/3.9.12
Date: Thu, 08 Sep 2022 18:03:35 GMT

{
  "contents": [
    "File contents for tsis2_sc_L2_v01_20220101_20220102.zip"
  ], 
  "filename": "tsis2_sc_L2_v01_20220101_20220102.zip"
}
HTTP/1.0 200 OK
Content-Type: application/json
Content-Length: 110
SDTP-TransactionID: 721
Server: Werkzeug/2.0.3 Python/3.9.12
Date: Thu, 08 Sep 2022 18:03:35 GMT

{
  "contents": [
    "File contents for tsis2_L1_20220104.zip"
  ], 
  "filename": "tsis2_L1_20220104.zip"
}
HTTP/1.0 200 OK
Content-Type: application/json
Content-Length: 154
SDTP-TransactionID: 722
Server: Werkzeug/2.0.3 Python/3.9.12
Date: Thu, 08 Sep 2022 18:03:36 GMT

{
  "contents": [
    "File contents for tsis2_ssi_L3_c24h_v01_20220103_20220104.txt"
  ], 
  "filename": "tsis2_ssi_L3_c24h_v01_20220103_20220104.txt"
}


In [14]:
# Delete the range of files
! curl -i -X DELETE "http://127.0.0.1:8000/files/1810-1812" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"

## Examples of bad/invalid requests

In [15]:
# Send request to endpoint that doesn't exist
# Expect a 404 response
! curl -i -X GET "http://127.0.0.1:8000/bogus_endpoint" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

404


In [18]:
# Try to register a closed account
# Expect a 401 response
! curl -i -X PUT "http://127.0.0.1:8000/register" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

401


In [19]:
# Request a file list with bogus parameters
# Expect a 400 response
! curl -i -X GET "http://127.0.0.1:8000/files?bogus_param=foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files?ShortName=TSIS_SC_L2&bogus_param=foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files?date=20210101" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

400
400
400


In [20]:
# Request a file list with valid parameters but no files returned
# Expect a 200 response with empty file list
! curl -i -X GET "http://127.0.0.1:8000/files?date=1984-04-04" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert"








{
  "files": []
}


In [21]:
# Request a file that doesn't exist
# Expect a 404 response
! curl -i -X GET "http://127.0.0.1:8000/files/1234567" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

404


In [22]:
# Request a file with invalid fileid
# Expect a 400 response
! curl -i -X GET "http://127.0.0.1:8000/files/-1" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files/1.5" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files/foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X GET "http://127.0.0.1:8000/files/9999999999999999" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

400
400
400
400


In [23]:
# Try to delete a file that doesn't exist
# Expect a 404 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/1234567" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

404


In [24]:
# Try to delete a file with invalid fileid
# Expect a 400 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/-1" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/1.5" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

400
400
400


In [25]:
# Try to delete a range of files that don't exist
# Expect a 404 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/1234567-1234569" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

404


In [26]:
# Try to delete a range of files with invalid fileids
# Expect a 400 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/foo-bar" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/10-5" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/123-foo" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/123 - 456" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s
! curl -i -X DELETE "http://127.0.0.1:8000/files/10-10" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

400
400
400
000
400


In [27]:
# Try to request a file that user doesn't have access to (file 23456 is hard-coded to be restricted)
# Expect a 403 response
! curl -i -X GET "http://127.0.0.1:8000/files/23456" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

403


In [28]:
# Try to delete a file that user doesn't have access to (file 23456 is hard-coded to be restricted)
# Expect a 403 response
! curl -i -X DELETE "http://127.0.0.1:8000/files/23456" -H "Accept: application/json" -H "Cert-UID: ges_disc_cert" -o /dev/null -w '%{http_code}\n' -s

403


In [ ]:
# Send A LOT of requests at once